# SFT UDS Pipeline — Real-Dataset Test

Order: **build → batching → real shards → training + benchmark**, on:

| Dataset | format | mapping |
|---|---|---|
| `tatsu-lab/alpaca` | `alpaca` | instruction(+input) → prompt, output → response |
| `HuggingFaceH4/ultrachat_200k` | `chat` | messages[user] → prompt, messages[assistant] → response |
| `CohereLabs/aya_dataset` | `pair` | inputs → prompt, targets → response |

Config lives in `datasets_real.json` (small `limit`s + streaming so it runs fast).
GPU strongly recommended for §4.

## 0 · Clean clone & locate
Deletes any stale `/content/sft`, re-clones fresh, then finds `bindings.cpp`.

In [ ]:
import os, sys, glob, shutil, subprocess

shutil.rmtree("/content/sft", ignore_errors=True)          # drop stale/partial copies
subprocess.run(["git","clone","--depth","1",
                "https://github.com/SniAssia/sft.git","/content/sft"], check=True)

def find_code_dir():
    for root, dirs, files in os.walk("/content/sft"):
        dirs[:] = [d for d in dirs if d != ".git"]
        if "bindings.cpp" in files:
            return root
    return None

CODE = find_code_dir()
assert CODE, "bindings.cpp not found — is the code committed under sft_final/?"
sys.path.insert(0, CODE)
print("code dir:", CODE)

# header completeness
needed = {"shard_format","sample","length_queues","shard_streamer","distributed","timer",
          "shard_reader","batch_scheduler","collator","prefetcher","data_pipeline"}
present = {os.path.basename(p)[:-4] for p in glob.glob(os.path.join(CODE,"*.hpp"))}
miss = sorted(needed - present)
assert not miss, f"repo missing headers: {miss} — commit the full sft_final bundle"

# parser-version guard (pair parser for aya lives in the updated prepare_shards.py)
import prepare_shards as ps
assert hasattr(ps.ZoneParser, "build"), \
    "old prepare_shards.py in repo — commit the updated one (adds the aya 'pair' parser)"
assert os.path.exists(os.path.join(CODE, "datasets_real.json")), \
    "datasets_real.json missing — commit it alongside prepare_shards.py"
print("headers + parser version OK ✅")

In [ ]:
# deps: torch preinstalled on Colab; add transformers + datasets
import importlib
def have(m):
    try: importlib.import_module(m); return True
    except Exception: return False
need=[m for m in ["torch","transformers","datasets"] if not have(m)]
if need: subprocess.run([sys.executable,"-m","pip","install","-q",*need], check=True)
import torch; print("torch",torch.__version__,"| CUDA:",torch.cuda.is_available())

## 1 · Build the C++ extension

In [ ]:
b = subprocess.run([sys.executable,"setup.py","build_ext","--inplace"],
                   cwd=CODE, capture_output=True, text=True)
print(b.stdout[-1200:])
if b.returncode!=0: print("STDERR:\n",b.stderr[-2500:]); raise RuntimeError("build failed")
import uds_loader
print("uds_loader OK:", [x for x in dir(uds_loader) if not x.startswith("_")])

## 2 · Batching test (model-free)
Synthetic ids — proves the C++ data path (shapes, masking, padding, Option-B window, formation timing). No downloads.

In [ ]:
import random, prepare_shards as ps
SYN=os.path.join(CODE,"_shards_synth"); MAXLEN=2048
cfg=ps.Config(out_dir=SYN,max_seq_length=MAXLEN,shard_size=500,band_short_max=512,band_medium_max=1536)
w=ps.ShardWriter(cfg); random.seed(0)
for _ in range(1500):
    r=random.random()
    t=(random.randint(20,500) if r<.4 else random.randint(512,1500) if r<.7
       else random.randint(1536,2048) if r<.9 else random.randint(2100,4000))
    pl,cl=max(1,t//5),t//5; rl=t-pl-cl; mk=lambda n:[random.randint(1,84000) for _ in range(n)]
    c,isk,bd=ps.decide_case(pl,cl,rl,MAXLEN); w.add(ps.TokSample(mk(pl),mk(cl),mk(rl),isk,c,bd))
w.flush(); print("bands S/M/L/C:",w.band_counts)
pc=uds_loader.PipelineConfig(); pc.shards=sorted(glob.glob(os.path.join(SYN,"shard_*.bin")))
pc.B=8; pc.chunked_rate=0.25; pc.pad_id=0; pc.option_b_window=MAXLEN; pc.pad_to_multiple=8; pc.num_epochs=1
pipe=uds_loader.DataPipeline(pc); pipe.start()
seen=0
for i in range(30):
    pool=pipe.next_pool()
    if pool is None: break
    ii,am,lb=pool.input_ids,pool.attention_mask,pool.labels
    assert ii.dtype==torch.int64 and ii.shape[1]%8==0 and (lb[am==0]==-100).all()
    if pool.is_chunked: assert ii.shape[1]==MAXLEN
    seen+=len(pool)
pipe.stop()
print(f"pooled {seen} samples | formation {pipe.formation_mean_ms():.3f}ms  stall {pipe.stall_mean_ms():.3f}ms")
print("BATCHING PASSED ✅")

## 3 · Real shards (alpaca + ultrachat_200k + aya)

Runs the offline stage on the three datasets in `datasets_real.json`. First run
downloads the datasets (streamed, capped by `limit`) and the Jais tokenizer.

`MAXLEN=1024` keeps §4 fast on CPU — set `2048` for a real run on GPU.

In [ ]:
MAXLEN = 1024                                   # use 2048 for real runs (GPU)
REAL   = os.path.join(CODE, "_shards_real")
r = subprocess.run([sys.executable,"prepare_shards.py",
        "--config", os.path.join(CODE,"datasets_real.json"),
        "--out", REAL,
        "--tokenizer","inceptionai/jais-family-590m",
        "--max-seq-length", str(MAXLEN),
        "--shard-size","1024","--workers","4"],
        cwd=CODE, capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode!=0: print("STDERR:\n",r.stderr[-3000:]); raise RuntimeError("shard prep failed")
import json
meta=json.load(open(os.path.join(REAL,"meta.json")))
print("\nmeta:",{k:meta[k] for k in ["vocab_size","pad_id","max_seq_length","num_samples","band_counts","source_stats"]})

## 4 · Training + benchmark
UDS-selected SFT on the real shards. Reports **batch formation / training / total** + prefetch overlap.

In [ ]:
from transformers import AutoModelForCausalLM
from uds import UDSConfig, UDSSelector
from benchmark import Benchmark
device="cuda" if torch.cuda.is_available() else "cpu"
STEPS,B,K,WARMUP=15,16,6,3
model=AutoModelForCausalLM.from_pretrained("inceptionai/jais-family-590m",
        trust_remote_code=True, torch_dtype=torch.float32).to(device); model.train()
opt=torch.optim.AdamW(model.parameters(),lr=1e-5)
sel=UDSSelector(int(meta["vocab_size"]),
        UDSConfig(K=K,alpha=1.0,svd_proj_dim=128,fp_dim1=64,fp_dim2=8,
                  start_sampling_step=WARMUP,device=device))
pc=uds_loader.PipelineConfig(); pc.shards=sorted(glob.glob(os.path.join(REAL,"shard_*.bin")))
pc.B=B; pc.homogeneous=True; pc.chunked_rate=0.05; pc.pad_id=int(meta["pad_id"])
pc.option_b_window=int(meta["max_seq_length"]); pc.pad_to_multiple=8
pc.num_epochs=-1; pc.prefetch_workers=3; pc.ring_capacity=4
pipe=uds_loader.DataPipeline(pc); pipe.start()
bench=Benchmark(sync=lambda: torch.cuda.synchronize() if torch.cuda.is_available() else None); bench.start()
for step in range(STEPS):
    with bench.phase("batch_formation"): pool=pipe.next_pool()
    if pool is None: break
    ii=pool.input_ids.to(device); am=pool.attention_mask.to(device); lb=pool.labels.to(device)
    if step<WARMUP or pool.is_chunked:
        t_ii,t_am,t_lb=ii,am,lb
    else:
        with bench.phase("uds_scoring",gpu=True):
            with torch.no_grad(): logits=model(input_ids=ii,attention_mask=am).logits
            s,z,si,se=sel.score(logits,am); idx=sel.select(s); sel.commit(z[idx])
        t_ii,t_am,t_lb=ii[idx],am[idx],lb[idx]
    with bench.phase("training",gpu=True):
        opt.zero_grad(set_to_none=True)
        out=model(input_ids=t_ii,attention_mask=t_am,labels=t_lb); out.loss.backward(); opt.step()
    bench.add_samples(t_ii.shape[0],tokens=int(t_am.sum().item()))
    print(f"step {step:2d} | pool={len(pool)} train={t_ii.shape[0]} band={pool.band} loss={out.loss.item():.4f}")
bench.stop(); pipe.stop()
cpp={"formation_total_s":pipe.formation_total_s(),"formation_mean_ms":pipe.formation_mean_ms(),
     "stall_total_s":pipe.stall_total_s(),"stall_mean_ms":pipe.stall_mean_ms()}
print(bench.report(cpp)); bench.save(os.path.join(CODE,"benchmark.json"),cpp)

## 5 · Multi-GPU (DDP)
```bash
torchrun --nproc_per_node=8 train.py --shards ./_shards_real \
    --model inceptionai/jais-family-590m --steps 200 --B 64 --K 32 --warmup 20
```
Each rank: same seeded shard order, own subset; UDS all-gathers scores for global
TopK and syncs the diversity buffer.